# Session 1 Lab — One Model, Three Servers 🏁
**EXL AI Services · Track 2.A · Model Serving Stacks**

In this lab we package the **same model** (`Qwen/Qwen2.5-0.5B-Instruct`) three ways — **FastAPI**, **BentoML**, **vLLM** — then benchmark **cold start**, **p95 latency** and **tokens/sec**, and pick a default stack with a written rationale.

### VM requirements
- Ubuntu 22.04, Python 3.10+, ~16 GB RAM, 20 GB disk
- **GPU (NVIDIA T4 or better) needed only for Step 3 (vLLM).** Steps 1–2 run fine on CPU.
- Ports 8001, 8002, 8003 free

> **How servers run inside Jupyter:** a server is a long-running process, so we launch each one as a *background subprocess* and talk to it with `requests`. Always run the **cleanup cell** at the end of each step before starting the next server.

## Step 0 — Setup (one time, ~5 min)

In [ ]:
# Install everything for Steps 1-2 (CPU-friendly)
%pip install -q fastapi uvicorn transformers torch bentoml requests pandas

# vLLM only if you have an NVIDIA GPU — uncomment:
# %pip install -q vllm

import torch
print("GPU available:", torch.cuda.is_available())

In [ ]:
# Shared helpers used by every step
import subprocess, time, requests, statistics, signal, os

PROCS = {}  # name -> Popen, so we can stop servers cleanly

def launch(name, cmd, health_url, timeout=600):
    """Start a server in the background and STOPWATCH its cold start."""
    print(f"Launching {name} ...")
    t0 = time.perf_counter()
    proc = subprocess.Popen(cmd, stdout=open(f"{name}.log", "w"),
                            stderr=subprocess.STDOUT, preexec_fn=os.setsid)
    PROCS[name] = proc
    while True:
        try:
            r = requests.get(health_url, timeout=2)
            if r.status_code < 500:
                break
        except Exception:
            pass
        if time.perf_counter() - t0 > timeout:
            raise RuntimeError(f"{name} did not come up — check {name}.log")
        time.sleep(0.5)
    cold = time.perf_counter() - t0
    print(f"✅ {name} ready — COLD START = {cold:.1f}s")
    return cold

def stop(name):
    """Kill a server and free its port."""
    p = PROCS.get(name)
    if p and p.poll() is None:
        os.killpg(os.getpgid(p.pid), signal.SIGTERM)
        p.wait(timeout=30)
    print(f"🛑 {name} stopped")

RESULTS = {}  # stack -> dict of benchmark numbers
print("Helpers ready.")

## Step 1 — Serve with FastAPI (10 min)
Our **baseline**: we write the server ourselves. Model loads ONCE at startup — never inside the endpoint (that's hiring a new chef for every order!).

In [ ]:
%%writefile app.py
# app.py — our own mini model server
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import pipeline

app = FastAPI()

# load ONCE at startup — never inside the endpoint!
generator = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")

class Prompt(BaseModel):
    text: str
    max_tokens: int = 64

@app.post("/generate")
def generate(req: Prompt):
    out = generator(req.text, max_new_tokens=req.max_tokens)
    return {"answer": out[0]["generated_text"]}

In [ ]:
# Launch it in the background (cold start is measured automatically)
cold = launch("fastapi",
              ["uvicorn", "app:app", "--port", "8001"],
              "http://localhost:8001/docs")
RESULTS["FastAPI"] = {"cold_start_s": round(cold, 1)}

In [ ]:
# First request — test like a real client
r = requests.post("http://localhost:8001/generate",
                  json={"text": "What is UPI?", "max_tokens": 64})
print(r.json()["answer"][:300])

**👀 While it runs, note down:** fire this cell 3–4 times — does the 2nd call feel faster than the 1st? (Warm-up, caches.) Also open `http://<vm-ip>:8001/docs` in a browser — free Swagger UI.

### Benchmark FastAPI — p50 / p95 / tokens/sec
Same exam for every server: **warm up first** (never benchmark a cold kitchen), then measure 50 requests.

In [ ]:
from transformers import AutoTokenizer
TOK = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
PROMPT = {"text": "Explain UPI in one line", "max_tokens": 64}

def benchmark(url, n=50, payload=PROMPT, extract=lambda j: j["answer"]):
    for _ in range(3):                      # --- warm-up ---
        requests.post(url, json=payload)
    lat, tokens, total_t = [], 0, 0.0        # --- measure ---
    for _ in range(n):
        t0 = time.perf_counter()
        resp = requests.post(url, json=payload).json()
        dt = time.perf_counter() - t0
        lat.append(dt); total_t += dt
        tokens += len(TOK.encode(extract(resp)))
    lat.sort()
    return {"p50_s": round(lat[n//2], 2),
            "p95_s": round(lat[int(n*0.95)-1], 2),
            "tokens_per_s": round(tokens/total_t, 1)}

RESULTS["FastAPI"].update(benchmark("http://localhost:8001/generate"))
RESULTS["FastAPI"]

In [ ]:
# Cleanup — free the port before the next server
stop("fastapi")

## Step 2 — Serve with BentoML (15 min)
Pack the model like a **tiffin box**: model + code + deps in one versioned artifact. Same model, same benchmark.

In [ ]:
%%writefile service.py
# service.py — the model's tiffin box
import bentoml
from transformers import pipeline

@bentoml.service(traffic={"timeout": 120})
class LLMService:
    def __init__(self):
        # runs once per worker at startup
        self.pipe = pipeline("text-generation",
                             model="Qwen/Qwen2.5-0.5B-Instruct")

    @bentoml.api
    def generate(self, text: str, max_tokens: int = 64) -> str:
        out = self.pipe(text, max_new_tokens=max_tokens)
        return out[0]["generated_text"]

In [ ]:
cold = launch("bentoml",
              ["bentoml", "serve", "service:LLMService", "--port", "8002"],
              "http://localhost:8002/healthz")
RESULTS["BentoML"] = {"cold_start_s": round(cold, 1)}

In [ ]:
# BentoML's API takes flat JSON args and returns a plain string
r = requests.post("http://localhost:8002/generate",
                  json={"text": "What is UPI?", "max_tokens": 64})
print(r.json()[:300] if isinstance(r.json(), str) else r.json())

In [ ]:
RESULTS["BentoML"].update(
    benchmark("http://localhost:8002/generate",
              extract=lambda j: j if isinstance(j, str) else str(j)))
RESULTS["BentoML"]

**👀 Observe:** compare `bentoml.log` with `fastapi.log` — BentoML spawns managed workers for you. Cold start is likely a bit slower: *packaging convenience has a price — measure it, don't guess it.* Homework: `bentoml build` then `bentoml containerize` → ready Docker image (we deploy it in Session 2).

In [ ]:
stop("bentoml")

## Step 3 — Serve with vLLM (10 min) — *GPU required*
Zero server code: one command gives you an OpenAI-compatible API with continuous batching + PagedAttention.

> **No GPU on this VM?** Skip to Step 4 and use the class's shared GPU numbers — the comparison discussion still works.

In [ ]:
cold = launch("vllm",
              ["vllm", "serve", "Qwen/Qwen2.5-0.5B-Instruct", "--port", "8003"],
              "http://localhost:8003/health", timeout=900)
RESULTS["vLLM"] = {"cold_start_s": round(cold, 1)}

In [ ]:
# OpenAI-style call — same client code works for OpenAI, Bedrock proxies, vLLM
VLLM_URL = "http://localhost:8003/v1/chat/completions"
VLLM_PAYLOAD = {"model": "Qwen/Qwen2.5-0.5B-Instruct",
                "max_tokens": 64,
                "messages": [{"role": "user", "content": "Explain UPI in one line"}]}
r = requests.post(VLLM_URL, json=VLLM_PAYLOAD)
print(r.json()["choices"][0]["message"]["content"][:300])

In [ ]:
RESULTS["vLLM"] = {**RESULTS["vLLM"],
    **benchmark(VLLM_URL, payload=VLLM_PAYLOAD,
                extract=lambda j: j["choices"][0]["message"]["content"])}
RESULTS["vLLM"]

In [ ]:
# 🔥 The moment of truth: 5 CONCURRENT requests — they DON'T queue one behind another.
# Continuous batching at work (try the same against FastAPI later and compare).
from concurrent.futures import ThreadPoolExecutor

def one_call(_):
    t0 = time.perf_counter()
    requests.post(VLLM_URL, json=VLLM_PAYLOAD)
    return round(time.perf_counter() - t0, 2)

with ThreadPoolExecutor(5) as ex:
    print("5 concurrent latencies:", list(ex.map(one_call, range(5))))

In [ ]:
stop("vllm")

## Step 4 — The scorecard 📊

In [ ]:
import pandas as pd
df = pd.DataFrame(RESULTS).T[["cold_start_s", "p50_s", "p95_s", "tokens_per_s"]]
df.columns = ["Cold start (s)", "p50 (s)", "p95 (s)", "Tokens/sec"]
df

**Expected pattern:** vLLM wins tokens/sec & p95 under load · FastAPI wins cold start · BentoML wins "easiest path to Docker".

## Step 5 — Pick your default & defend it ✍️
Fill in the rationale (this is the actual lab deliverable — numbers without a decision = incomplete lab):

> *"We choose ____ as our default because our workload is ____. In our benchmark it gave p95 = ____, tokens/sec = ____, cold start = ____. We accept the trade-off of ____, and we would revisit this choice if ____."*

All four must be present: **numbers, workload, trade-off, revisit condition.**

In [ ]:
# Final cleanup — stop anything still running
for name in list(PROCS):
    stop(name)